<a href="https://colab.research.google.com/github/ShadiFarzankia/Human-Value-Detection/blob/master/Human_Value_Detection_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Importing the Libararies

In [6]:
!pip install -q transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 74.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.8/236.8 kB 22.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 78.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 43.6 MB/s eta 0:00:00


In [7]:
import numpy as np
import pandas as pd
from sklearn import metrics
import transformers
import torch
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, accuracy_score, f1_score

In [29]:
# Random seed to repeat experiments.
RANDOM_SEED = 42
transformers.set_seed(RANDOM_SEED)

### Loading the Data

In [19]:
#loading the data into variables
train_args = pd.read_csv("arguments-training.tsv",delimiter='\t')
valid_args = pd.read_csv("arguments-validation.tsv",delimiter='\t')
test_args = pd.read_csv("arguments-test.tsv",delimiter='\t')

train_labels = pd.read_csv("labels-training.tsv",delimiter='\t')
valid_labels = pd.read_csv("labels-validation.tsv",delimiter='\t')
test_labels = pd.read_csv("labels-test.tsv",delimiter='\t')

In [21]:
#checking the size of data
print("The size of train_args:", train_args.shape, "  The size of train_labes:", train_labels.shape)

print("The size of valid_args:", valid_args.shape, "  The size of valid_labes:", valid_labels.shape)

print("The size of test_args:", test_args.shape, "  The size of test_labes:", test_labels.shape)

The size of train_args: (5393, 4)   The size of train_labes: (5393, 21)
The size of valid_args: (1896, 4)   The size of valid_labes: (1896, 21)
The size of test_args: (1576, 4)   The size of test_labes: (1576, 21)


In [26]:
print(train_args.head())

  Argument ID                                   Conclusion       Stance  \
0      A01002                  We should ban human cloning  in favor of   
1      A01005                      We should ban fast food  in favor of   
2      A01006  We should end the use of economic sanctions      against   
3      A01007         We should abolish capital punishment      against   
4      A01008                We should ban factory farming      against   

                                             Premise  
0  we should ban human cloning as it will only ca...  
1  fast food should be banned because it is reall...  
2  sometimes economic sanctions are the only thin...  
3  capital punishment is sometimes the only optio...  
4  factory farming allows for the production of c...  


In [28]:
print(train_labels.head())

  Argument ID  Self-direction: thought  Self-direction: action  Stimulation  \
0      A01002                        0                       0            0   
1      A01005                        0                       0            0   
2      A01006                        0                       0            0   
3      A01007                        0                       0            0   
4      A01008                        0                       0            0   

   Hedonism  Achievement  Power: dominance  Power: resources  Face  \
0         0            0                 0                 0     0   
1         0            0                 0                 0     0   
2         0            0                 1                 0     0   
3         0            0                 0                 0     0   
4         0            0                 0                 0     0   

   Security: personal  ...  Tradition  Conformity: rules  \
0                   0  ...          0       

In [31]:
#meging the arguments and labels
training_full_data = pd.merge(train_args, train_labels, on='Argument ID')

In [32]:
print(training_full_data.head())

  Argument ID                                   Conclusion       Stance  \
0      A01002                  We should ban human cloning  in favor of   
1      A01005                      We should ban fast food  in favor of   
2      A01006  We should end the use of economic sanctions      against   
3      A01007         We should abolish capital punishment      against   
4      A01008                We should ban factory farming      against   

                                             Premise  Self-direction: thought  \
0  we should ban human cloning as it will only ca...                        0   
1  fast food should be banned because it is reall...                        0   
2  sometimes economic sanctions are the only thin...                        0   
3  capital punishment is sometimes the only optio...                        0   
4  factory farming allows for the production of c...                        0   

   Self-direction: action  Stimulation  Hedonism  Achievement 